# Experiment 6: AutoGluon time-series market trend + tabular residuals

Tree ensembles are strong at property-level interpolation but are not reliable trend extrapolators. This experiment forecasts a dense monthly market series for each `TOWN × FLAT_TYPE` segment, then uses AutoGluon Tabular to model each listing's log-rent relative to that forecast. The final prediction is `market forecast × exp(property residual)`.

The validation split is chronological: the last 12 labelled months are held out. The final forecast origin is the first test month, so no labelled rents from the forecast horizon are used by the market model.

In [1]:
# If needed: pip install autogluon.timeseries autogluon.tabular
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from autogluon.common import space
from autogluon.tabular import TabularPredictor
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from preprocessing import RentalPreprocessor, load_datasets

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
VALIDATION_HORIZON = 12
MIN_HISTORY_MONTHS = 24
HPO_TRIALS = 25
TS_TRIAL_TIME_LIMIT = int(os.environ.get('EXP6_TS_TRIAL_TIME_LIMIT', '120'))
TS_FINAL_TIME_LIMIT = int(os.environ.get('EXP6_TS_FINAL_TIME_LIMIT', '600'))
TABULAR_HPO_TIME_LIMIT = int(os.environ.get('EXP6_TABULAR_HPO_TIME_LIMIT', '3600'))
TABULAR_FINAL_TIME_LIMIT = int(os.environ.get('EXP6_TABULAR_FINAL_TIME_LIMIT', '3600'))
HPO_KWARGS = {'num_trials': HPO_TRIALS, 'scheduler': 'local', 'searcher': 'random'}


In [2]:
def canonical_flat_type(values):
    """Make `3 room` (train) and `3-room` (test) the same segment."""
    clean = values.fillna('unknown').astype(str).str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)
    return clean.replace({'multigeneration': 'multigeneration'})

def add_segment_and_period(frame):
    result = frame.copy()
    result['period'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    town = result['TOWN'].fillna('unknown').astype(str).str.strip().str.lower()
    result['market_segment'] = town + ' | ' + canonical_flat_type(result['FLAT_TYPE'])
    return result

data_dir, datasets = load_datasets(project_root / 'data')
raw_train = add_segment_and_period(datasets['train'])
raw_test = add_segment_and_period(datasets['test'])
forecast_origin = raw_test['period'].min()
available_train = raw_train[raw_train['period'] < forecast_origin].copy()
if available_train.empty:
    raise ValueError('No labelled months precede the first test month.')

all_train_periods = np.sort(available_train['period'].unique())
validation_start = pd.Timestamp(all_train_periods[-VALIDATION_HORIZON])
history_raw = available_train[available_train['period'] < validation_start].copy()
validation_raw = available_train[available_train['period'] >= validation_start].copy()

print(f'All labelled history used for final forecast: {available_train.period.min():%Y-%m} to {available_train.period.max():%Y-%m}')
print(f'Forecast origin: {forecast_origin:%Y-%m}; test horizon ends: {raw_test.period.max():%Y-%m}')
print(f'Validation: train through {(validation_start - pd.offsets.MonthBegin(1)):%Y-%m}; holdout from {validation_start:%Y-%m}')


All labelled history used for final forecast: 2021-01 to 2025-02
Forecast origin: 2025-03; test horizon ends: 2026-07
Validation: train through 2024-02; holdout from 2024-03


In [3]:
# Candidate covariates must be present for *every* forecast month.
# COE values and stock values are supplied in the project auxiliary files through 2026-07.
def build_monthly_covariates(datasets):
    month_number = {name: number for number, name in enumerate(
        ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], 1
    )}
    coe = datasets['coe'].copy()
    coe['period'] = pd.to_datetime(dict(
        year=coe['YEAR'], month=coe['MONTH'].str.lower().map(month_number), day=1
    ))
    for column in ['QUOTA_PREMIUM', 'PREVAILING_QUOTA_PREMIUM', 'QUOTA', 'BIDS_RECEIVED']:
        coe[column] = pd.to_numeric(
            coe[column].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
            errors='coerce',
        )
    coe_monthly = coe.groupby('period', as_index=False).agg(
        coe_quota_premium_mean=('QUOTA_PREMIUM', 'mean'),
        coe_prevailing_quota_premium_mean=('PREVAILING_QUOTA_PREMIUM', 'mean'),
        coe_quota_mean=('QUOTA', 'mean'),
        coe_bids_mean=('BIDS_RECEIVED', 'mean'),
    )
    stock = datasets['stock'].copy()
    stock['period'] = pd.to_datetime(stock['DATE']).dt.to_period('M').dt.to_timestamp()
    stock_monthly = stock.groupby('period', as_index=False).agg(
        stock_close_mean=('CLOSE', 'mean'),
        stock_close_median=('CLOSE', 'median'),
        stock_volume_mean=('VOLUME', 'mean'),
    )
    result = coe_monthly.merge(stock_monthly, on='period', how='outer').sort_values('period')
    result['month_sin'] = np.sin(2 * np.pi * result['period'].dt.month / 12)
    result['month_cos'] = np.cos(2 * np.pi * result['period'].dt.month / 12)
    return result

monthly_covariates = build_monthly_covariates(datasets)
candidate_covariates = [
    'coe_quota_premium_mean', 'coe_prevailing_quota_premium_mean', 'coe_quota_mean', 'coe_bids_mean',
    'stock_close_mean', 'stock_close_median', 'stock_volume_mean',
]
market_for_analysis = available_train.groupby('period', as_index=False)[TARGET].mean().rename(columns={TARGET: 'market_rent'})
correlation_base = market_for_analysis.merge(monthly_covariates, on='period', how='left').sort_values('period')
correlation_base['log_market_rent'] = np.log(correlation_base['market_rent'])
correlation_base['monthly_log_rent_change'] = correlation_base['log_market_rent'].diff()
correlation_rows = []
for feature in candidate_covariates:
    for lag_months in (0, 1, 3, 6):
        values = correlation_base[feature].shift(lag_months)
        paired = pd.DataFrame({
            'log_market_rent': correlation_base['log_market_rent'],
            'monthly_log_rent_change': correlation_base['monthly_log_rent_change'],
            feature: values,
        }).dropna()
        correlation_rows.append({
            'feature': feature, 'lag_months': lag_months, 'n_months': len(paired),
            'level_correlation': paired['log_market_rent'].corr(paired[feature]),
            'change_correlation': paired['monthly_log_rent_change'].corr(paired[feature]),
        })
correlation_report = pd.DataFrame(correlation_rows).sort_values('change_correlation', key=lambda s: s.abs(), ascending=False)
correlation_path = experiment_dir / 'exp_6_covariate_correlations.csv'
correlation_report.to_csv(correlation_path, index=False)
display(correlation_report)
print(f'Saved covariate correlations to {correlation_path}')

# Selection after checking the report: bids predicts monthly changes, premium is a useful level proxy,
# and sin/cos expose calendar seasonality. Quota is excluded because it is nearly collinear with bids;
# stock aggregates are excluded because their change correlations are weak and unstable.
KNOWN_COVARIATES = ['coe_bids_mean', 'coe_quota_premium_mean', 'month_sin', 'month_cos']
required_periods = pd.date_range(available_train['period'].min(), raw_test['period'].max(), freq='MS')
covariate_coverage = monthly_covariates.set_index('period').reindex(required_periods)[KNOWN_COVARIATES]
if covariate_coverage.isna().any().any():
    raise ValueError(f'Missing known-covariate values: {covariate_coverage.isna().sum().to_dict()}')
print('Selected known covariates:', KNOWN_COVARIATES)


,feature,lag_months,n_months,level_correlation,change_correlation
12,coe_bids_mean,0,49,-0.013767,-0.419916
8,coe_quota_mean,0,49,-0.115834,-0.407143
7,coe_prevailing_quota_premium_mean,6,44,0.946004,-0.390066
13,coe_bids_mean,1,49,-0.151219,-0.369908
9,coe_quota_mean,1,49,-0.258741,-0.364655
3,coe_quota_premium_mean,6,44,0.930753,-0.351106
6,coe_prevailing_quota_premium_mean,3,47,0.961676,-0.319496
22,stock_close_median,3,47,0.195332,0.279248
20,stock_close_median,0,49,0.007782,0.259783
14,coe_bids_mean,3,47,-0.254031,-0.258512


Saved covariate correlations to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_6_covariate_correlations.csv
Selected known covariates: ['coe_bids_mean', 'coe_quota_premium_mean', 'month_sin', 'month_cos']


In [4]:
def monthly_market(frame):
    monthly = (frame.groupby(['market_segment', 'period'], as_index=False)[TARGET].mean()
        .rename(columns={'market_segment': 'item_id', 'period': 'timestamp', TARGET: 'target'})
        .sort_values(['item_id', 'timestamp']))
    eligible = monthly.groupby('item_id')['target'].count()
    monthly = monthly[monthly['item_id'].isin(eligible[eligible >= MIN_HISTORY_MONTHS].index)]
    return monthly.merge(monthly_covariates[['period', *KNOWN_COVARIATES]], left_on='timestamp', right_on='period', how='left').drop(columns='period')

def as_ts(frame):
    return TimeSeriesDataFrame.from_data_frame(frame, id_column='item_id', timestamp_column='timestamp')

def market_actual(frame):
    return (frame.groupby(['market_segment', 'period'], as_index=False)[TARGET].mean()
        .rename(columns={TARGET: 'actual_market_rent'}))

def make_future_known_covariates(predictor, history_ts):
    future = predictor.make_future_data_frame(history_ts).reset_index()
    future = future.merge(monthly_covariates[['period', *KNOWN_COVARIATES]], left_on='timestamp', right_on='period', how='left').drop(columns='period')
    if future[KNOWN_COVARIATES].isna().any().any():
        raise ValueError('A selected known covariate is unavailable in the forecast horizon.')
    return as_ts(future[['item_id', 'timestamp', *KNOWN_COVARIATES]])

def sample_ts_params(rng):
    return {
        'num_leaves': int(rng.integers(16, 129)),
        'num_boost_round': int(rng.integers(200, 1501)),
        'max_depth': int(rng.integers(3, 11)),
        'learning_rate': float(10 ** rng.uniform(np.log10(0.01), np.log10(0.20))),
        'min_data_in_leaf': int(rng.integers(10, 151)),
        'feature_fraction': float(rng.uniform(0.60, 1.00)),
        'bagging_fraction': float(rng.uniform(0.60, 1.00)),
    }

def ts_hyperparameters(params):
    # Recursive/Direct Tabular use LightGBM internally. num_boost_round is the iteration count.
    return {
        'RecursiveTabular': {'model_name': 'GBM', 'model_hyperparameters': dict(params)},
        'DirectTabular': {'model_name': 'GBM', 'model_hyperparameters': dict(params)},
        'Theta': {},
        'ETS': {},
    }

def fit_market_model(history, prediction_length, model_path, params, time_limit):
    history_ts = as_ts(monthly_market(history))
    predictor = TimeSeriesPredictor(
        target='target', prediction_length=prediction_length, freq='MS', eval_metric='RMSE',
        known_covariates_names=KNOWN_COVARIATES, path=str(model_path),
    )
    predictor.fit(
        train_data=history_ts, hyperparameters=ts_hyperparameters(params),
        enable_ensemble=True, num_val_windows=2, time_limit=time_limit,
        random_seed=RANDOM_STATE, verbosity=2,
    )
    return predictor, predictor.predict(history_ts, known_covariates=make_future_known_covariates(predictor, history_ts))

def score_market_forecast(forecast, actual_rows):
    predicted = forecast.reset_index()[['item_id', 'timestamp', 'mean']].rename(
        columns={'item_id': 'market_segment', 'timestamp': 'period', 'mean': 'market_rent_forecast'}
    )
    scored = market_actual(actual_rows).merge(predicted, on=['market_segment', 'period'], how='inner')
    return mean_squared_error(scored['actual_market_rent'], scored['market_rent_forecast']) ** 0.5

# Manual outer-loop HPO gives each of the 25 configurations the same future 12-month backtest.
# It avoids relying on nested search-space support in TimeSeriesPredictor.
rng = np.random.default_rng(RANDOM_STATE)
ts_hpo_rows = []
for trial in range(1, HPO_TRIALS + 1):
    params = sample_ts_params(rng)
    _, trial_forecast = fit_market_model(
        history_raw, VALIDATION_HORIZON, experiment_dir / 'ag_exp6_ts_hpo' / f'trial_{trial:02d}',
        params, TS_TRIAL_TIME_LIMIT,
    )
    ts_hpo_rows.append({'trial': trial, 'market_RMSE': score_market_forecast(trial_forecast, validation_raw), **params})
ts_hpo_results = pd.DataFrame(ts_hpo_rows).sort_values('market_RMSE').reset_index(drop=True)
ts_hpo_path = experiment_dir / 'exp_6_timeseries_hpo_results.csv'
ts_hpo_results.to_csv(ts_hpo_path, index=False)
display(ts_hpo_results.head(10))
print(f'Saved {len(ts_hpo_results)} time-series trials to {ts_hpo_path}')
BEST_TS_PARAMS = ts_hpo_results.iloc[0].drop(labels=['trial', 'market_RMSE']).to_dict()
BEST_TS_PARAMS = {key: int(value) if key in {'num_leaves', 'num_boost_round', 'max_depth', 'min_data_in_leaf'} else float(value) for key, value in BEST_TS_PARAMS.items()}

validation_ts_predictor, validation_market_forecast = fit_market_model(
    history_raw, VALIDATION_HORIZON, experiment_dir / 'ag_exp6_ts_validation_hpo',
    BEST_TS_PARAMS, TS_FINAL_TIME_LIMIT,
)
display(validation_ts_predictor.leaderboard(silent=True))

validation_market = validation_market_forecast.reset_index()[['item_id', 'timestamp', 'mean']].rename(
    columns={'item_id': 'market_segment', 'timestamp': 'period', 'mean': 'market_rent_forecast'}
)
validation_with_market = validation_raw.merge(validation_market, on=['market_segment', 'period'], how='left')
print('Validation listings covered by a market forecast:', validation_with_market.market_rent_forecast.notna().mean())


Beginning AutoGluon training... Time limit = 120s
AutoGluon will save models to '/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_ts_hpo/trial_01'
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.90 GB / 16.00 GB (30.6%)
Disk Space Avail:   197.27 GB / 460.38 GB (42.8%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': {'DirectTabular': {'model_hyperparameters': {'bagging_fraction': 0.6376709391550598,
                                                                 'feature_fraction'

,trial,market_RMSE,num_leaves,num_boost_round,max_depth,learning_rate,min_data_in_leaf,feature_fraction,bagging_fraction
0,23,273.239648,30,792,8,0.025007,38,0.831688,0.670709
1,3,273.280502,72,682,4,0.068816,140,0.929105,0.777366
2,7,273.639107,120,1168,5,0.026540,146,0.748184,0.787822
3,4,273.711508,66,495,3,0.012107,88,0.931052,0.852666
4,5,273.838858,34,1186,8,0.183192,59,0.957248,0.911353
5,11,274.096465,110,460,9,0.105636,11,0.865940,0.882066
6,16,274.107157,112,566,10,0.072640,51,0.822813,0.913559
7,17,274.511769,128,1064,6,0.114568,67,0.666789,0.609085
8,10,274.851048,65,1246,9,0.023720,64,0.872998,0.655901
9,1,275.367054,26,1206,8,0.130937,71,0.878947,0.637671


Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_ts_validation_hpo'
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.23 GB / 16.00 GB (26.4%)
Disk Space Avail:   197.22 GB / 460.38 GB (42.8%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': {'DirectTabular': {'model_hyperparameters': {'bagging_fraction': 0.6707091131756927,
                                                                 'feature_fractio

Saved 25 time-series trials to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_6_timeseries_hpo_results.csv


	-458.1650     = Validation score (-RMSE)
	1.12    s     = Training runtime
	0.02    s     = Validation (prediction) runtime
Training timeseries model DirectTabular. Training for up to 149.7s of the 598.7s of remaining time.
	-802.9600     = Validation score (-RMSE)
	1.32    s     = Training runtime
	0.02    s     = Validation (prediction) runtime
Training timeseries model ETS. Training for up to 199.1s of the 597.4s of remaining time.
	-519.1526     = Validation score (-RMSE)
	0.05    s     = Training runtime
	0.17    s     = Validation (prediction) runtime
Training timeseries model Theta. Training for up to 298.6s of the 597.2s of remaining time.
	-409.9331     = Validation score (-RMSE)
	0.06    s     = Training runtime
	0.04    s     = Validation (prediction) runtime
Fitting 1 ensemble(s), in 1 layers.
Training ensemble model WeightedEnsemble. Training for up to 597.1s.
	Ensemble weights: {'RecursiveTabular': 0.23, 'Theta': 0.77}
	-404.2505     = Validation score (-RMSE)
	0.11    s

,model,score_val,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-404.250545,0.060942,0.108194,5
1,Theta,-409.933106,0.039860,0.056015,4
2,RecursiveTabular,-458.164971,0.020686,1.122381,1
3,ETS,-519.152596,0.171024,0.051750,3
4,DirectTabular,-802.960007,0.017151,1.319275,2


Validation listings covered by a market forecast: 0.99708297932597


In [5]:
def attach_observed_market_residual(frame, market_source):
    market = (market_source.groupby(['market_segment', 'period'], as_index=False)[TARGET].mean()
        .rename(columns={TARGET: 'observed_market_rent'}))
    result = frame.merge(market, on=['market_segment', 'period'], how='left')
    result['LOG_RENT_RESIDUAL'] = np.log(result[TARGET]) - np.log(result['observed_market_rent'])
    return result

def prepare_tabular_features(frame):
    result = frame.drop(columns=[TARGET, 'LOG_RENT_RESIDUAL', 'observed_market_rent', 'market_segment', 'period'], errors='ignore').copy()
    result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    return result

# The residual labels are relative to the observed market at training time.
# At inference, the observed market is replaced by the time-series forecast.
residual_train_raw = attach_observed_market_residual(history_raw, history_raw)
residual_valid_raw = attach_observed_market_residual(validation_raw, validation_raw)

validation_preprocessor = RentalPreprocessor(datasets)
residual_train = validation_preprocessor.fit_transform(residual_train_raw)
residual_valid = validation_preprocessor.transform(residual_valid_raw)
X_residual_train = prepare_tabular_features(residual_train)
X_residual_valid = prepare_tabular_features(residual_valid)

# 25 random-search trials per model family. The GBM search tunes leaves, boosting rounds
# (iterations), tree depth and learning rate; CatBoost provides a complementary tree family.
TABULAR_HYPERPARAMETERS = {
    'GBM': {
        'num_leaves': space.Int(16, 128),
        'num_boost_round': space.Int(200, 1500),
        'max_depth': space.Int(3, 12),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'min_data_in_leaf': space.Int(10, 150),
        'feature_fraction': space.Real(0.60, 1.00),
        'bagging_fraction': space.Real(0.60, 1.00),
    },
    'CAT': {
        'iterations': space.Int(200, 1500),
        'depth': space.Int(4, 10),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'l2_leaf_reg': space.Real(1.0, 30.0, log=True),
    },
}

residual_predictor = TabularPredictor(
    label='LOG_RENT_RESIDUAL', problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp6_residual_validation'),
).fit(
    train_data=X_residual_train.assign(LOG_RENT_RESIDUAL=residual_train['LOG_RENT_RESIDUAL'].to_numpy()),
    tuning_data=X_residual_valid.assign(LOG_RENT_RESIDUAL=residual_valid['LOG_RENT_RESIDUAL'].to_numpy()),
    hyperparameters=TABULAR_HYPERPARAMETERS, hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality', time_limit=TABULAR_HPO_TIME_LIMIT, verbosity=2,
)
display(residual_predictor.leaderboard(silent=True, extra_info=True))

valid_residual_prediction = residual_predictor.predict(X_residual_valid).to_numpy()
validation_with_market['residual_prediction'] = valid_residual_prediction
validation_with_market['prediction'] = (
    validation_with_market['market_rent_forecast'] * np.exp(validation_with_market['residual_prediction'])
)
scored = validation_with_market.dropna(subset=['prediction'])
display(pd.DataFrame([{
    'RMSE': mean_squared_error(scored[TARGET], scored['prediction']) ** 0.5,
    'MAE': mean_absolute_error(scored[TARGET], scored['prediction']),
    'R2': r2_score(scored[TARGET], scored['prediction']),
    'Scored listings': len(scored),
}]))


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.14 GB / 16.00 GB (25.9%)
Disk Space Avail:   197.22 GB / 460.38 GB (42.8%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_residual_validation"
Train Data Rows:    111658
Train Data Columns: 33
Tuning Data Rows:    35310
Tuning Data Columns: 33
Label Column:       LOG_RENT_RESIDUAL
Problem Type:       regression
Preprocessing data...
Using Fe

[1000]	valid_set's rmse: 0.181323


 56%|█████▌    | 14/25 [00:30<00:35,  3.19s/it]

[1000]	valid_set's rmse: 0.181426


100%|██████████| 25/25 [01:04<00:00,  2.57s/it]
Fitted model: LightGBM/T1 ...
	-0.1835	 = Validation score   (-root_mean_squared_error)
	0.52s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	-0.1814	 = Validation score   (-root_mean_squared_error)
	1.15s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	-0.1814	 = Validation score   (-root_mean_squared_error)
	0.73s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	-0.1813	 = Validation score   (-root_mean_squared_error)
	1.9s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	-0.1818	 = Validation score   (-root_mean_squared_error)
	0.71s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: LightGBM/T6 ...
	-0.1813	 = Validation score   (-root_mean_squared_error)
	1.33s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T7 ...
	-0.1814	 = Validation score   (-root_mean_squa

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order,...,hyperparameters,hyperparameters_fit,ag_args_fit,features,compile_time,child_hyperparameters,child_hyperparameters_fit,child_ag_args_fit,ancestors,descendants
0,WeightedEnsemble_L2,-0.179918,root_mean_squared_error,0.700126,162.078636,0.000345,0.035074,2,True,51,...,"{'use_orig_features': False, 'valid_stacker': ...",{},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[CatBoost/T8, LightGBM/T21, CatBoost/T18, Ligh...",None,"{'ensemble_size': 25, 'subsample_size': 1000000}",{'ensemble_size': 18},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[CatBoost/T8, LightGBM/T21, CatBoost/T18, Ligh...",[]
1,CatBoost/T9,-0.180353,root_mean_squared_error,0.046356,23.489505,0.046356,23.489505,1,True,34,...,"{'iterations': 1489, 'learning_rate': 0.097046...",{'iterations': 1114},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
2,CatBoost/T18,-0.180376,root_mean_squared_error,0.076837,37.573131,0.076837,37.573131,1,True,43,...,"{'iterations': 1355, 'learning_rate': 0.057148...",{'iterations': 1339},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
3,CatBoost/T2,-0.180399,root_mean_squared_error,0.042024,17.598876,0.042024,17.598876,1,True,27,...,"{'iterations': 759, 'learning_rate': 0.0608433...",{'iterations': 706},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
4,CatBoost/T15,-0.180413,root_mean_squared_error,0.065444,31.661095,0.065444,31.661095,1,True,40,...,"{'iterations': 1343, 'learning_rate': 0.029239...",{'iterations': 1343},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
5,CatBoost/T3,-0.180419,root_mean_squared_error,0.056052,25.050864,0.056052,25.050864,1,True,28,...,"{'iterations': 1233, 'learning_rate': 0.031629...",{'iterations': 1233},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
6,CatBoost/T12,-0.180458,root_mean_squared_error,0.074365,36.128035,0.074365,36.128035,1,True,37,...,"{'iterations': 1277, 'learning_rate': 0.019464...",{'iterations': 1274},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[]
7,CatBoost/T8,-0.180464,root_mean_squared_error,0.045856,22.987587,0.045856,22.987587,1,True,33,...,"{'iterations': 859, 'learning_rate': 0.0866093...",{'iterations': 500},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
8,CatBoost/T4,-0.180474,root_mean_squared_error,0.055904,32.635080,0.055904,32.635080,1,True,29,...,"{'iterations': 800, 'learning_rate': 0.0418269...",{'iterations': 735},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]
9,CatBoost/T19,-0.180498,root_mean_squared_error,0.046845,29.637695,0.046845,29.637695,1,True,44,...,"{'iterations': 1367, 'learning_rate': 0.064744...",{'iterations': 474},"{'max_memory_usage_ratio': 1.0, 'max_gpu_memor...","[FLOOR_AREA_SQM, LEASE_COMMENCE_DATE, approval...",None,None,None,None,[],[WeightedEnsemble_L2]


,RMSE,MAE,R2,Scored listings
0,486.051843,352.387857,0.461511,35207


In [6]:
# Refit on every labelled month strictly before the first test month.
final_preprocessor = RentalPreprocessor(datasets)
final_train_raw = attach_observed_market_residual(available_train, available_train)
final_train = final_preprocessor.fit_transform(final_train_raw)
final_test = final_preprocessor.transform(raw_test)

X_final_train = prepare_tabular_features(final_train)
X_final_test = prepare_tabular_features(final_test)
final_residual_predictor = TabularPredictor(
    label='LOG_RENT_RESIDUAL', problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp6_residual_final'),
).fit(
    train_data=X_final_train.assign(LOG_RENT_RESIDUAL=final_train['LOG_RENT_RESIDUAL'].to_numpy()),
    hyperparameters=TABULAR_HYPERPARAMETERS, hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality', time_limit=TABULAR_FINAL_TIME_LIMIT, verbosity=2,
)

test_periods = pd.date_range(forecast_origin, raw_test['period'].max(), freq='MS')
final_ts_predictor, final_market_forecast = fit_market_model(
    available_train, len(test_periods), experiment_dir / 'ag_exp6_ts_final_hpo',
    BEST_TS_PARAMS, TS_FINAL_TIME_LIMIT,
)
test_market = final_market_forecast.reset_index()[['item_id', 'timestamp', 'mean']].rename(
    columns={'item_id': 'market_segment', 'timestamp': 'period', 'mean': 'market_rent_forecast'}
)
test_with_market = raw_test.merge(test_market, on=['market_segment', 'period'], how='left')
test_residual_prediction = final_residual_predictor.predict(X_final_test).to_numpy()
test_with_market['PREDICTED_MONTHLY_RENT'] = (
    test_with_market['market_rent_forecast'] * np.exp(test_residual_prediction)
)

# A new town/type segment cannot be forecast; use the prior global market mean.
fallback = available_train[TARGET].mean()
test_with_market['PREDICTED_MONTHLY_RENT'] = test_with_market['PREDICTED_MONTHLY_RENT'].fillna(fallback).clip(lower=0)
submission = test_with_market.drop(columns=['period', 'market_segment', 'market_rent_forecast'])
output_path = experiment_dir / 'exp_6_autogluon_timeseries_predictions.csv'
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission):,} predictions to {output_path}')
display(submission[['RENT_APPROVAL_DATE', 'TOWN', 'FLAT_TYPE', 'PREDICTED_MONTHLY_RENT']].head())


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.40 GB / 16.00 GB (27.5%)
Disk Space Avail:   195.41 GB / 460.38 GB (42.4%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp6_residual_final"
Train Data Rows:    146968
Train Data Columns: 33
Label Column:       LOG_RENT_RESIDUAL
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting Auto

[1000]	valid_set's rmse: 0.185816


100%|██████████| 25/25 [01:08<00:00,  2.76s/it]
Fitted model: LightGBM/T1 ...
	-0.1881	 = Validation score   (-root_mean_squared_error)
	0.55s	 = Training   runtime
	0.0s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	-0.1857	 = Validation score   (-root_mean_squared_error)
	1.37s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	-0.1856	 = Validation score   (-root_mean_squared_error)
	0.81s	 = Training   runtime
	0.0s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	-0.1857	 = Validation score   (-root_mean_squared_error)
	1.88s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	-0.1859	 = Validation score   (-root_mean_squared_error)
	0.79s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T6 ...
	-0.1856	 = Validation score   (-root_mean_squared_error)
	1.25s	 = Training   runtime
	0.0s	 = Validation runtime
Fitted model: LightGBM/T7 ...
	-0.1857	 = Validation score   (-root_mean_square

Saved 50,000 predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_6_autogluon_timeseries_predictions.csv


,RENT_APPROVAL_DATE,TOWN,FLAT_TYPE,PREDICTED_MONTHLY_RENT
0,2025-03,kallang/whampoa,3-room,2597.509690
1,2025-03,bukit timah,4-room,3219.667855
2,2025-03,pasir ris,4-room,3051.873320
3,2025-03,punggol,4-room,3064.971929
4,2025-03,tampines,4-room,3231.460298


In [7]:
# Competition submission format: one prediction per original test row.
# Run this after the preceding cell has created `test_with_market`.
competition_submission = pd.DataFrame({
    'Id': raw_test.index.to_numpy(),
    'Predicted': test_with_market['PREDICTED_MONTHLY_RENT'].to_numpy(),
})
assert competition_submission['Id'].is_unique
assert len(competition_submission) == len(raw_test)
assert competition_submission['Predicted'].notna().all()

competition_output_path = experiment_dir / 'exp_6_autogluon_timeseries_submission.csv'
competition_submission.to_csv(competition_output_path, index=False)
print(f'Saved {len(competition_submission):,} competition-format predictions to {competition_output_path}')
display(competition_submission.head())


Saved 50,000 competition-format predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_6_autogluon_timeseries_submission.csv


,Id,Predicted
0,0,2597.509690
1,1,3219.667855
2,2,3051.873320
3,3,3064.971929
4,4,3231.460298
